# M5 thesis — V2 evaluation of the 30,490 products (corrected pipeline)
Same pipeline code as the V2 tuning notebook. Every product: 10 models × 5 test windows (days 1,746–1,885) + 1 holdout window
(days 1,886–1,913), each product's history starting at its first sale. Results are saved in chunks of 250 products:
the run resumes automatically after a disconnection. Then: final table, MCS, characteristics (days 1–1,745), control run.

In [1]:
# Cell 1 — Setup
from google.colab import drive
drive.mount('/content/drive')
!pip install catboost arch -q
import os, sys, json, time, glob, platform, warnings
warnings.filterwarnings('ignore')
D = '/content/drive/MyDrive/M5_Thesis'
V2 = f'{D}/results/v2'
EV = f'{V2}/eval'; os.makedirs(EV, exist_ok=True)


Mounted at /content/drive


In [2]:
# Cell 2 — Pipeline V2 definitions (same code as in the tuning notebook)
# ============================================================================
# PIPELINE V2 — shared definitions (identical in the tuning and evaluation notebooks)
# ============================================================================
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

N_DAYS     = 1913                       # sales_train_validation.csv only
MAX_LAG    = 42
FIRST_ROW_DAY = MAX_LAG + 1             # row r forecasts day r + 43 (1-based)
H          = 28                         # window length (days)
CUT        = 1745                       # last day used for classes and characteristics
TUNE_STARTS = [1662, 1690, 1718]        # tuning windows: days 1,662-1,745 (before any test day)
EVAL_STARTS = [1746, 1774, 1802, 1830, 1858]   # 5 test windows: days 1,746-1,885
HOLD_START  = 1886                      # holdout window: days 1,886-1,913
MODELS = ['LGBM', 'XGBOOST', 'CATBOOST', 'HGBR', 'RANDOM_FOREST',
          'ELASTIC_NET', 'EXTRA_TREES', 'GRADIENT_BOOST', 'RIDGE', 'LASSO']
LINEAR = {'RIDGE', 'LASSO', 'ELASTIC_NET'}
N_TRIALS = {'RIDGE': 30, 'LASSO': 30, 'ELASTIC_NET': 30, 'RANDOM_FOREST': 30, 'EXTRA_TREES': 30,
            'LGBM': 50, 'XGBOOST': 50, 'CATBOOST': 50, 'HGBR': 50, 'GRADIENT_BOOST': 50}

def build_features(series):
    """Row for target day s uses only days before s. Lags 7..42 fall on the SAME weekday as s.
    Rolling mean/std (population std, ddof=0) over the w days before s. Vectorised with cumulative sums."""
    x = np.asarray(series, float)
    s_idx = np.arange(MAX_LAG, len(x))
    c1 = np.r_[0.0, np.cumsum(x)]; c2 = np.r_[0.0, np.cumsum(x * x)]
    cols = [x[s_idx - k] for k in (7, 14, 28, 35, 42)]
    def mean_w(w): return (c1[s_idx] - c1[s_idx - w]) / w
    def std_w(w):
        m = mean_w(w); v = (c2[s_idx] - c2[s_idx - w]) / w - m * m
        return np.sqrt(np.clip(v, 0, None))
    cols += [mean_w(w) for w in (7, 14, 28)] + [std_w(w) for w in (7, 28)]
    return np.column_stack(cols), x[s_idx]

def first_sale_row(y):
    """First row whose target is a sale (the product's history starts at its first sale, as in the M5 RMSSE)."""
    nz = np.flatnonzero(y > 0)
    return int(nz[0]) if len(nz) else len(y)

def window_rows(start_day):
    """Training rows = all target days before start_day; test rows = the 28 days from start_day."""
    a = start_day - FIRST_ROW_DAY
    return a, a + H                      # train = rows[:a], test = rows[a:a+H]

def rmsse(y_test, pred, y_train):
    """RMSSE with the M5 scale: mean squared one-step change of the training targets from the first sale onward."""
    t = y_train[first_sale_row(y_train):]
    if len(t) < 2: return np.nan
    scale = np.mean(np.diff(t) ** 2)
    return np.nan if scale == 0 else float(np.sqrt(np.mean((y_test - pred) ** 2) / scale))

def window_ok(y, start_day, min_rows=50):
    """A window is evaluable if the product has >= min_rows training rows after its first sale and a non-zero scale."""
    a, _ = window_rows(start_day)
    t = y[first_sale_row(y[:a]):a]
    return len(t) >= min_rows and np.any(np.diff(t) != 0)

def build_model(name, p, n_jobs=-1):
    p = dict(p)
    if name == 'LGBM':          return lgb.LGBMRegressor(objective='regression', verbosity=-1, n_jobs=n_jobs, random_state=42, n_estimators=200, subsample_freq=1, **p)
    if name == 'XGBOOST':       return XGBRegressor(objective='reg:squarederror', verbosity=0, n_jobs=n_jobs, random_state=42, n_estimators=200, **p)
    if name == 'CATBOOST':      return CatBoostRegressor(verbose=False, random_seed=42, iterations=200, thread_count=n_jobs, allow_writing_files=False, **p)
    if name == 'HGBR':          return HistGradientBoostingRegressor(random_state=42, max_iter=200, **p)
    if name == 'RANDOM_FOREST': return RandomForestRegressor(random_state=42, n_jobs=n_jobs, **p)
    if name == 'EXTRA_TREES':   return ExtraTreesRegressor(random_state=42, n_jobs=n_jobs, **p)
    if name == 'GRADIENT_BOOST':return GradientBoostingRegressor(random_state=42, n_estimators=200, **p)
    if name == 'ELASTIC_NET':   return ElasticNet(max_iter=10000, random_state=42, **p)
    if name == 'LASSO':         return Lasso(max_iter=10000, random_state=42, **p)
    if name == 'RIDGE':         return Ridge(**p)

def fit_predict(name, p, X, y, start_day, n_jobs=-1):
    """Train on the product's rows from its first sale up to the day before start_day; forecast the 28-day window."""
    a, b = window_rows(start_day)
    f = first_sale_row(y[:a])
    Xtr, ytr, Xte, yte = X[f:a], y[f:a], X[a:b], y[a:b]
    if name in LINEAR:
        sc = StandardScaler(); Xtr = sc.fit_transform(Xtr); Xte = sc.transform(Xte)
    m = build_model(name, p, n_jobs); m.fit(Xtr, ytr)
    pred = np.clip(m.predict(Xte), 0, None)
    return yte, pred, ytr

def search_space(name, trial):          # identical ranges to the July tuning (cells 85-105)
    s = trial.suggest_float; i = trial.suggest_int
    if name == 'LASSO':  return {'alpha': s('alpha', 1e-4, 100.0, log=True)}
    if name == 'RIDGE':  return {'alpha': s('alpha', 1e-4, 1000.0, log=True)}
    if name == 'ELASTIC_NET': return {'alpha': s('alpha', 1e-4, 100.0, log=True), 'l1_ratio': s('l1_ratio', 0.0, 1.0)}
    if name == 'LGBM':   return {'learning_rate': s('learning_rate', 0.01, 0.30), 'num_leaves': i('num_leaves', 8, 256),
                                 'min_child_samples': i('min_child_samples', 5, 200), 'reg_lambda': s('reg_lambda', 1e-3, 10.0, log=True),
                                 'subsample': s('subsample', 0.5, 1.0), 'colsample_bytree': s('colsample_bytree', 0.5, 1.0)}
    if name == 'XGBOOST':return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 3, 10),
                                 'min_child_weight': i('min_child_weight', 5, 100), 'reg_lambda': s('reg_lambda', 1e-3, 10.0, log=True),
                                 'subsample': s('subsample', 0.5, 1.0), 'colsample_bytree': s('colsample_bytree', 0.5, 1.0)}
    if name == 'CATBOOST': return {'learning_rate': s('learning_rate', 0.01, 0.30), 'depth': i('depth', 3, 10),
                                   'l2_leaf_reg': s('l2_leaf_reg', 1e-3, 10.0, log=True)}
    if name == 'HGBR':   return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 2, 15),
                                 'min_samples_leaf': i('min_samples_leaf', 5, 200), 'l2_regularization': s('l2_regularization', 1e-3, 10.0, log=True)}
    if name == 'GRADIENT_BOOST': return {'learning_rate': s('learning_rate', 0.01, 0.30), 'max_depth': i('max_depth', 2, 8),
                                         'subsample': s('subsample', 0.5, 1.0), 'min_samples_leaf': i('min_samples_leaf', 5, 100)}
    if name in ('RANDOM_FOREST', 'EXTRA_TREES'):
        return {'n_estimators': i('n_estimators', 50, 300), 'max_depth': i('max_depth', 3, 30),
                'max_features': s('max_features', 0.3, 1.0), 'min_samples_leaf': i('min_samples_leaf', 5, 100)}

def sbc_class(series_until_cut):
    """SBC (2005) on the product's history from its first sale: p = T / N (T = days since first sale, N = days with sales),
    CV2 of non-zero demand (sample variance, ddof=1); cut-offs 1.32 and 0.49. One sale only -> CV2 = 0. Fewer than 50 days since the first sale -> SHORT_HISTORY."""
    y = np.asarray(series_until_cut, float); nz = np.flatnonzero(y > 0)
    if len(nz) == 0: return 'NO_SALE', np.nan, np.nan
    y = y[nz[0]:]; pos = y[y > 0]
    p = len(y) / len(pos)
    if len(y) < 50: return 'SHORT_HISTORY', p, np.nan       # fewer than 50 days since the first sale
    cv2 = pos.var(ddof=1) / pos.mean() ** 2 if len(pos) > 1 else 0.0
    cls = ('SMOOTH' if cv2 <= 0.49 else 'ERRATIC') if p <= 1.32 else ('INTERMITTENT' if cv2 <= 0.49 else 'LUMPY')
    return cls, p, cv2


In [3]:
# Cell 3 — Data, classes, tuned settings
sales = pd.read_csv(f'{D}/data/sales_train_validation.csv').set_index('id')
day_cols = [c for c in sales.columns if c.startswith('d_')]
assert len(day_cols) == N_DAYS, 'Wrong file: expected 1,913 days'
Y = sales[day_cols].to_numpy(float)
cls = pd.read_csv(f'{V2}/classes_cut1745.csv').set_index('SKU_ID')['class_cut1745']
chk = np.array([sbc_class(y[:CUT])[0] for y in Y])
assert (cls.reindex(sales.index).values == chk).all(), 'classes_cut1745.csv does not match the pipeline: rerun Cells 1-4 of the tuning notebook'
PARAMS = json.load(open(f'{V2}/optimal_hyperparameters_v2.json'))
WINDOWS = EVAL_STARTS + [HOLD_START]                       # windows 1-5 = test, 6 = holdout
import hashlib
MANIFEST = {'params_sha1': hashlib.sha1(json.dumps(PARAMS, sort_keys=True).encode()).hexdigest(), 'windows': WINDOWS, 'cut': CUT}
mf = f'{EV}/manifest.json'
if os.path.exists(mf):
    assert json.load(open(mf)) == MANIFEST, 'Existing chunk files were made with other settings: move EV folder away before running'
else:
    json.dump(MANIFEST, open(mf, 'w'), indent=1)
missing = [(c, m) for c in ['INTERMITTENT', 'LUMPY', 'SMOOTH', 'ERRATIC'] for m in MODELS if m not in PARAMS.get(c, {})]
assert not missing, f'Tuned settings missing for: {missing}'
# Products without a tuning class (no sale or < 50 days of history by day 1,745) use the INTERMITTENT settings (largest class)
PARAM_CLASS = cls.where(cls.isin(['INTERMITTENT', 'LUMPY', 'SMOOTH', 'ERRATIC']), 'INTERMITTENT')
print(f'✅ {Y.shape[0]:,} products × {Y.shape[1]:,} days | settings for {sorted(PARAMS)}')
print('Products using INTERMITTENT settings by default:', int((PARAM_CLASS != cls).sum()), cls[PARAM_CLASS != cls].value_counts().to_dict())


✅ 30,490 products × 1,913 days | settings for ['ERRATIC', 'INTERMITTENT', 'LUMPY', 'SMOOTH']
Products using INTERMITTENT settings by default: 89 {'SHORT_HISTORY': 53, 'NO_SALE': 36}


In [ ]:
# Cell 4 — Evaluation of all products (about 3–4 days in total; resumes automatically)
import pyarrow  # parquet support
from joblib import Parallel, delayed
def eval_product(sku, series, params):
    X, y = build_features(series)
    res, fc = [], []
    for w, d0 in enumerate(WINDOWS, 1):
        if not window_ok(y, d0):
            res.append((sku, 'ALL', w, np.nan, 'skipped: history too short or constant')); continue
        for m in MODELS:
            try:
                yte, pred, ytr = fit_predict(m, params[m], X, y, d0, n_jobs=1)
                res.append((sku, m, w, rmsse(yte, pred, ytr), 'ok'))
                fc.append(pd.DataFrame({'SKU_ID': sku, 'Model': m, 'Window': np.int8(w),
                                        'Day': np.arange(d0, d0 + H, dtype=np.int16),
                                        'Actual': yte.astype(np.float32), 'Predicted': pred.astype(np.float32)}))
            except Exception as e:
                res.append((sku, m, w, np.nan, f'error: {type(e).__name__}'))
    return res, fc
CHUNK = 250
n_chunks = int(np.ceil(len(sales) / CHUNK)); t0 = time.time(); done_now = 0
for k in range(n_chunks):
    f_res, f_fc = f'{EV}/rmsse_{k:04d}.csv', f'{EV}/forecasts_{k:04d}.parquet'
    if os.path.exists(f_res) and os.path.exists(f_fc): continue
    idx = range(k * CHUNK, min((k + 1) * CHUNK, len(sales)))
    out = Parallel(n_jobs=-1)(delayed(eval_product)(sales.index[i], Y[i], PARAMS[PARAM_CLASS[sales.index[i]]]) for i in idx)
    fcs_all = [f for _, fcs in out for f in fcs]
    FC = pd.concat(fcs_all) if fcs_all else pd.DataFrame(columns=['SKU_ID', 'Model', 'Window', 'Day', 'Actual', 'Predicted'])
    FC.to_parquet(f_fc + '.tmp', index=False); os.replace(f_fc + '.tmp', f_fc)
    pd.DataFrame([r for res, _ in out for r in res], columns=['SKU_ID', 'Model', 'Window', 'RMSSE', 'Status']).to_csv(f_res + '.tmp', index=False)
    os.replace(f_res + '.tmp', f_res)
    done_now += 1; el = (time.time() - t0) / 3600
    left = n_chunks - len(glob.glob(f'{EV}/rmsse_*.csv'))
    print(f'✅ chunk {k + 1}/{n_chunks} saved — {el:.1f} h this session — about {el / done_now * left:.0f} h left')
print('🎉 EVALUATION FINISHED')


✅ chunk 102/122 saved — 0.6 h this session — about 12 h left
✅ chunk 103/122 saved — 1.1 h this session — about 11 h left
✅ chunk 104/122 saved — 1.7 h this session — about 10 h left
✅ chunk 105/122 saved — 2.3 h this session — about 10 h left
✅ chunk 106/122 saved — 2.8 h this session — about 9 h left
✅ chunk 107/122 saved — 3.4 h this session — about 9 h left
✅ chunk 108/122 saved — 4.0 h this session — about 8 h left
✅ chunk 109/122 saved — 4.7 h this session — about 8 h left
✅ chunk 110/122 saved — 5.3 h this session — about 7 h left
✅ chunk 111/122 saved — 5.8 h this session — about 6 h left
✅ chunk 112/122 saved — 6.3 h this session — about 6 h left
✅ chunk 113/122 saved — 6.9 h this session — about 5 h left
✅ chunk 114/122 saved — 7.5 h this session — about 5 h left
✅ chunk 115/122 saved — 8.1 h this session — about 4 h left
✅ chunk 116/122 saved — 8.6 h this session — about 3 h left
✅ chunk 117/122 saved — 9.2 h this session — about 3 h left
✅ chunk 118/122 saved — 9.8 h this s

In [ ]:
# Cell 5 — Final tables: RMSSE per product and model (test windows 1-5, holdout window 6) + quality checks
R = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{EV}/rmsse_*.csv'))], ignore_index=True)
assert R['SKU_ID'].nunique() == len(sales), 'Some chunks are missing: rerun Cell 4'
print('Status counts:', R['Status'].value_counts().to_dict())
ok = R[R['Status'] == 'ok']
full = ok.groupby(['SKU_ID', 'Window'])['Model'].transform('size') == len(MODELS)
print(f"Windows removed because not all 10 models succeeded: {ok.loc[~full, ['SKU_ID', 'Window']].drop_duplicates().shape[0]}")
ok = ok[full]
test = ok[ok['Window'] <= 5].pivot_table(index='SKU_ID', columns='Model', values='RMSSE', aggfunc='mean')
nwin = ok[ok['Window'] <= 5].groupby(['SKU_ID', 'Model'])['RMSSE'].count().unstack()
hold = ok[ok['Window'] == 6].pivot_table(index='SKU_ID', columns='Model', values='RMSSE', aggfunc='mean')
final = test[MODELS].copy()
final.insert(0, 'n_test_windows', nwin.min(axis=1).reindex(final.index))
final.insert(0, 'Demand_Type', cls.reindex(final.index))
final['Best_Model'] = final[MODELS].idxmin(axis=1); final['Best_RMSSE'] = final[MODELS].min(axis=1)
final.to_csv(f'{V2}/final_eval_v2.csv')
hold[MODELS].to_csv(f'{V2}/holdout_rmsse_v2.csv')
print(f'Products with a test result: {len(final):,} of {len(sales):,}')
print('Number of test windows used per product:', final['n_test_windows'].value_counts().sort_index().to_dict())
print('Products with a holdout result:', len(hold))
print('Products where a model failed in at least one window:', R.loc[R['Status'].str.startswith('error'), 'SKU_ID'].nunique())
print(final['Best_Model'].value_counts().to_string())
print('✅ Saved final_eval_v2.csv and holdout_rmsse_v2.csv')


Status counts: {'ok': 1827150, 'skipped: history too short or constant': 225}
Windows removed because not all 10 models succeeded: 0
Products with a test result: 30,474 of 30,490
Number of test windows used per product: {1: 8, 2: 4, 3: 31, 4: 31, 5: 30400}
Products with a holdout result: 30482
Products where a model failed in at least one window: 0
Best_Model
RIDGE             5212
EXTRA_TREES       5116
RANDOM_FOREST     4186
ELASTIC_NET       3171
LASSO             3051
CATBOOST          2824
LGBM              2109
XGBOOST           1885
HGBR              1688
GRADIENT_BOOST    1232
✅ Saved final_eval_v2.csv and holdout_rmsse_v2.csv


In [ ]:
# Cell 6 — Model Confidence Set on the 5 test windows (Hansen, Lunde & Nason, 2011): 90%, block 7, 300 reps (about 2-3 h)
from arch.bootstrap import MCS
from joblib import Parallel, delayed
MC = f'{V2}/mcs'; os.makedirs(MC, exist_ok=True)
def mcs_one(sku, g, series):
    _, y = build_features(series)
    g = g[g['Window'] <= 5].copy()
    def scale(w):                                        # same scale as the RMSSE of that window
        a, _ = window_rows(WINDOWS[w - 1]); t = y[first_sale_row(y[:a]):a]; return np.mean(np.diff(t) ** 2)
    g['loss'] = (g['Actual'] - g['Predicted']) ** 2 / g['Window'].map({w: scale(w) for w in g['Window'].unique()})
    L = g.pivot_table(index=['Window', 'Day'], columns='Model', values='loss', observed=True).dropna()
    if L.empty: return None
    uniq = L.T.drop_duplicates().T
    twin = {m: next(u for u in uniq.columns if np.allclose(L[m], uniq[u])) for m in L.columns}
    status = 'mcs'
    try:
        if uniq.shape[1] == 1: pv = pd.Series(1.0, index=uniq.columns)
        else:
            mc = MCS(uniq, size=0.10, reps=300, block_size=7, method='R', seed=0); mc.compute(); pv = mc.pvalues['Pvalue']
    except Exception:
        ml = uniq.mean(); pv = pd.Series(np.where(ml <= ml.min() * (1 + 1e-9), 1.0, 0.0), index=uniq.columns); status = 'tie'
    row = {'SKU_ID': sku, 'mcs_status': status}
    for a in (0.05, 0.10, 0.25):                           # one run gives the 95%, 90% and 75% sets
        inc = set(pv.index[pv > a])
        for m in MODELS: row[f'MCS{int(100 - a * 100)}_{m}'] = int(m in twin and twin[m] in inc)
    row['mcs_size'] = sum(row[f'MCS90_{m}'] for m in MODELS)
    return row
t0 = time.time(); files = sorted(glob.glob(f'{EV}/forecasts_*.parquet'))
for k, f in enumerate(files):
    out = f"{MC}/mcs_{os.path.basename(f).split('_')[1].split('.')[0]}.csv"
    if os.path.exists(out): continue
    F = pd.read_parquet(f)
    rows = [r for r in Parallel(n_jobs=-1)(delayed(mcs_one)(s, g, Y[sales.index.get_loc(s)]) for s, g in F.groupby('SKU_ID')) if r]
    pd.DataFrame(rows).to_csv(out + '.tmp', index=False); os.replace(out + '.tmp', out)
    if k % 10 == 0: print(f'✅ chunk {k + 1}/{len(files)} — {(time.time() - t0) / 60:.0f} min')
M = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{MC}/mcs_*.csv')) if os.path.getsize(f) > 1], ignore_index=True)
M.to_csv(f'{V2}/mcs_v2.csv', index=False)
print(f'🎉 MCS finished: {len(M):,} products | mean set size {M.mcs_size.mean():.2f} | ties {int((M.mcs_status == "tie").sum())}')
print('Note: the MCS pools squared scaled errors over the 5 windows; the bootstrap blocks may cross window boundaries.')


✅ chunk 1/122 — 0 min
✅ chunk 11/122 — 3 min
✅ chunk 21/122 — 6 min
✅ chunk 31/122 — 9 min
✅ chunk 41/122 — 12 min
✅ chunk 51/122 — 15 min
✅ chunk 61/122 — 18 min
✅ chunk 71/122 — 20 min
✅ chunk 81/122 — 23 min
✅ chunk 91/122 — 26 min
✅ chunk 101/122 — 29 min
✅ chunk 111/122 — 32 min
✅ chunk 121/122 — 34 min
🎉 MCS finished: 30,474 products | mean set size 8.92 | ties 0
Note: the MCS pools squared scaled errors over the 5 windows; the bootstrap blocks may cross window boundaries.


In [ ]:
# Cell 7 — Demand characteristics on days 1-1,745 (same definitions as in October, cut-off moved to 1,745)
CUT_F = 1745                                 # last day before the first test day (1,746)
cols = day_cols[:CUT_F]
XC = sales[cols].to_numpy(float)

# ---- Part 1: demand characteristics (history up to day 1,746) ----
rows = []
for sku, y in zip(sales.index, XC):
    T = len(y)
    nz = np.flatnonzero(y > 0)
    first = nz[0] if len(nz) else T
    a = y[first:]
    pos = y[y > 0]
    gaps = np.diff(nz) if len(nz) > 1 else np.array([np.nan])
    z = (a == 0).astype(int)
    e = np.diff(np.r_[0, z, 0])
    runs = np.flatnonzero(e == -1) - np.flatnonzero(e == 1)
    runs = runs if len(runs) else np.array([0])
    wk = np.array([a[k::7].mean() for k in range(7)]) if len(a) >= 14 else np.full(7, np.nan)
    last, before = a[-365:], a[:-365]
    rows.append({
        'SKU_ID': sku,
        'first_sale_day': first + 1,
        'active_days': len(a),
        'n_sales_days': len(nz),
        'total_units': y.sum(),
        'zero_share_all': (y == 0).mean(),
        'zero_share_active': (a == 0).mean() if len(a) else np.nan,
        'longest_zero_run': runs.max(),
        'mean_zero_run': runs.mean(),
        'ADI_active': np.nanmean(gaps),
        'CV_intervals': gaps.std() / gaps.mean() if len(nz) > 2 else np.nan,
        'mean_demand_active': a.mean() if len(a) else np.nan,
        'mean_size': pos.mean() if len(pos) else np.nan,
        'median_size': np.median(pos) if len(pos) else np.nan,
        'min_size': pos.min() if len(pos) else np.nan,
        'max_demand': y.max(),
        'CV2_size': pos.var(ddof=1) / pos.mean()**2 if len(pos) > 1 else np.nan,   # ddof=1, as in sbc_class
        'peak_ratio': y.max() / pos.mean() if len(pos) else np.nan,
        'recent_mean_28': a[-28:].mean() if len(a) else np.nan,
        'recent_zero_share_91': (a[-91:] == 0).mean() if len(a) else np.nan,
        'trend_ratio': last.mean() / before.mean() if len(before) and before.mean() > 0 else np.nan,
        'weekday_strength': wk.std() / wk.mean() if np.nanmean(wk) > 0 else np.nan,
        'autocorr_lag1': np.corrcoef(a[:-1], a[1:])[0, 1] if len(a) > 2 and a.std() > 0 else np.nan,
    })
feat = pd.DataFrame(rows)

# ---- Part 2: price and calendar characteristics (same cut-off) ----
cal = pd.read_csv(f'{D}/data/calendar.csv').iloc[:CUT_F]
prices = pd.read_csv(f'{D}/data/sell_prices.csv')
prices['key'] = prices['item_id'] + '_' + prices['store_id']
keys = (sales['item_id'] + '_' + sales['store_id']).values
weeks = pd.unique(cal['wm_yr_wk'])
Pw = prices.pivot(index='key', columns='wm_yr_wk', values='sell_price').reindex(index=keys, columns=weeks).to_numpy(np.float32)
day_w = pd.Index(weeks).get_indexer(cal['wm_yr_wk'])
Pday = Pw[:, day_w]
YC = XC.astype(np.float32)
listed = ~np.isnan(Pday)

def masked_mean(A, m):
    n = m.sum(1)
    return np.where(n > 0, (A * m).sum(1) / np.maximum(n, 1), np.nan)

first = np.where((YC > 0).any(1), (YC > 0).argmax(1), YC.shape[1])
active = np.arange(YC.shape[1])[None, :] >= first[:, None]
event = cal['event_name_1'].notna().to_numpy()[None, :]
weekend = cal['weekday'].isin(['Saturday', 'Sunday']).to_numpy()[None, :]
snap_by_state = cal[['snap_CA', 'snap_TX', 'snap_WI']].to_numpy().T.astype(bool)
snap = snap_by_state[sales['state_id'].map({'CA': 0, 'TX': 1, 'WI': 2}).to_numpy()]
Yw = pd.DataFrame(YC).T.groupby(day_w).sum().T.to_numpy(np.float32)
corr = np.full(len(YC), np.nan)
for i in range(len(YC)):
    ok = ~np.isnan(Pw[i])
    if ok.sum() > 10 and Pw[i, ok].std() > 0 and Yw[i, ok].std() > 0:
        corr[i] = np.corrcoef(Pw[i, ok], Yw[i, ok])[0, 1]

with np.errstate(all='ignore'):
    regular = np.nanmax(Pw, 1)
    new = pd.DataFrame({
        'SKU_ID': sales.index,
        'listed_share': listed.mean(1),
        'first_listed_day': np.where(listed.any(1), listed.argmax(1) + 1, np.nan),
        'zero_share_listed': masked_mean((YC == 0).astype(np.float32), listed),
        'mean_price': np.nanmean(Pw, 1),
        'price_cv': np.nanstd(Pw, 1) / np.nanmean(Pw, 1),
        'n_price_changes': np.nansum(np.abs(np.diff(Pw, axis=1)) > 0.001, 1),
        'discount_share': np.nanmean(np.where(np.isnan(Pw), np.nan, Pw < 0.95 * regular[:, None]), 1),
        'price_sales_corr': corr,
        'event_uplift': masked_mean(YC, active & event) / masked_mean(YC, active & ~event),
        'snap_uplift': masked_mean(YC, active & snap) / masked_mean(YC, active & ~snap),
        'weekend_share': (YC * weekend).sum(1) / np.maximum(YC.sum(1), 1),
    })

out = feat.merge(new, on='SKU_ID')
out.to_csv(f'{D}/results/v2/sku_features_cut1745.csv', index=False)
print('✅ Saved:', out.shape, '| products with no sale before the cut-off:', int((out['n_sales_days'] == 0).sum()))


✅ Saved: (30490, 34) | products with no sale before the cut-off: 36


In [ ]:
# Cell 8 — Control run: all 10 models with DEFAULT settings on 500 test products (about 1-2 h)
# 'Default' = library defaults plus the pipeline's fixed values (200 trees for boosting, seeds).
smp = pd.read_csv(f'{D}/results/FINAL_SAMPLE_3305.csv')['id']
ctrl_ids = smp.sample(500, random_state=0).tolist()
def eval_default(sku, series):
    X, y = build_features(series); out = []
    for w, d0 in enumerate(EVAL_STARTS, 1):
        if not window_ok(y, d0): continue
        for m in MODELS:
            try:
                yte, pred, ytr = fit_predict(m, {}, X, y, d0, n_jobs=1); out.append((sku, m, w, rmsse(yte, pred, ytr)))
            except Exception:
                out.append((sku, m, w, np.nan))
    return out
CTRL_F = f'{V2}/control_default_settings.csv'
from joblib import Parallel, delayed
if not os.path.exists(CTRL_F):
    rows = [r for res in Parallel(n_jobs=-1)(delayed(eval_default)(s, Y[sales.index.get_loc(s)]) for s in ctrl_ids) for r in res]
    pd.DataFrame(rows, columns=['SKU_ID', 'Model', 'Window', 'RMSSE']).to_csv(CTRL_F, index=False)
C = pd.read_csv(CTRL_F).pivot_table(index='SKU_ID', columns='Model', values='RMSSE', aggfunc='mean')[MODELS]
T = pd.read_csv(f'{V2}/final_eval_v2.csv').set_index('SKU_ID').loc[C.index, MODELS]
cmp = pd.DataFrame({'Median RMSSE tuned': T.median(), 'Median RMSSE default': C.median(),
                    'Avg rank tuned': T.rank(axis=1).mean(), 'Avg rank default': C.rank(axis=1).mean(),
                    'Win share tuned': T.idxmin(axis=1).value_counts(normalize=True).reindex(MODELS).fillna(0),
                    'Win share default': C.idxmin(axis=1).value_counts(normalize=True).reindex(MODELS).fillna(0)}).round(4)
from scipy import stats
print(cmp.sort_values('Avg rank tuned').to_string())
print(f"Spearman correlation of the model ranking, tuned vs default: {stats.spearmanr(cmp['Avg rank tuned'], cmp['Avg rank default'])[0]:.3f}")
cmp.to_csv(f'{V2}/control_tuned_vs_default.csv'); print('✅ Saved control_tuned_vs_default.csv')


                Median RMSSE tuned  Median RMSSE default  Avg rank tuned  Avg rank default  Win share tuned  Win share default
EXTRA_TREES                 0.6826                0.7630           4.348             6.071            0.170              0.034
RANDOM_FOREST               0.6836                0.7451           4.852             5.357            0.148              0.050
ELASTIC_NET                 0.6851                0.6983           5.046             3.636            0.120              0.296
LASSO                       0.6838                0.7022           5.122             4.156            0.120              0.020
CATBOOST                    0.6843                0.7366           5.134             4.962            0.088              0.044
RIDGE                       0.6837                0.6857           5.190             2.306            0.146              0.496
HGBR                        0.6862                0.7714           5.982             7.018            0.058    